# axon-lang — Treine sua própria IA em português (Colab Free)

Fine-tune (QLoRA) de um modelo aberto usando as suas lições de `treinamento_portugues/`.
Roda numa GPU **T4 grátis**. O resultado é **seu** e roda offline depois.

**Antes de começar:** menu `Ambiente de execução → Alterar tipo de ambiente → GPU (T4)`.

Passos: 1) instalar · 2) carregar modelo · 3) enviar seus dados · 4) preparar · 5) treinar · 6) testar · 7) salvar no Drive.

## 1. Instalar (Unsloth = QLoRA rápido e leve)

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

## 2. Carregar um modelo aberto (4-bit, cabe na T4)
Troque `MODEL` se quiser (Qwen2.5-3B, Mistral-7B, Gemma-2-2B...). 3B é o ponto doce na T4 grátis.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"   # bom em PT; alternativas: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
MAX_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_LEN, load_in_4bit=True,
)
# adaptadores LoRA (só ~1% dos pesos treina -> barato e rápido)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

## 3. Enviar seus dados
No seu PC, **compacte a pasta** `treinamento_portugues` num `.zip` e envie aqui.

In [ ]:
from google.colab import files
import zipfile, os

up = files.upload()                      # selecione treinamento_portugues.zip
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall("dados")
# acha a pasta raiz das lições
BASE = None
for root, dirs, fs in os.walk("dados"):
    if os.path.basename(root) == "treinamento_portugues":
        BASE = root; break
BASE = BASE or "dados"
print("pasta:", BASE)

## 4. Preparar o dataset (lição -> par pergunta/resposta)
Cada `.md` vira um exemplo: **pergunta** = "Explique: <título>" · **resposta** = corpo da lição.
Assim o modelo aprende a *responder* de forma didática em português, não só a completar texto.

In [ ]:
import glob, re
from datasets import Dataset

def parse(fp):
    with open(fp, encoding="utf-8") as f:
        txt = f.read().strip()
    linhas = [l for l in txt.splitlines()]
    titulo = next((l.lstrip('# ').strip() for l in linhas if l.startswith('#')), None)
    corpo = txt
    if titulo is None:
        # usa a estrutura de pastas como assunto
        titulo = fp.replace('\\','/').split('/')[-2].replace('_',' ')
    return titulo, corpo

exemplos = []
for fp in glob.glob(os.path.join(BASE, '**', '*.md'), recursive=True):
    titulo, corpo = parse(fp)
    if len(corpo) < 200:
        continue
    msgs = [
        {"role": "user", "content": f"Explique de forma didática: {titulo}"},
        {"role": "assistant", "content": corpo},
    ]
    exemplos.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False)})

print(f"{len(exemplos)} exemplos de treino")
ds = Dataset.from_list(exemplos)

## 5. Treinar (QLoRA)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=2, learning_rate=2e-4,
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="cosine", seed=42, output_dir="saida",
        report_to="none",
    ),
)
trainer.train()

## 6. Testar a sua IA

In [ ]:
FastLanguageModel.for_inference(model)

def responder(pergunta):
    msgs = [{"role": "user", "content": pergunta}]
    inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                           return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

print(responder("O que é o determinante de uma matriz?"))
print("---")
print(responder("Explique a Revolução Francesa."))

## 7. Salvar no Google Drive (pra não perder quando a sessão cair)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
model.save_pretrained('/content/drive/MyDrive/axon_ia_lora')       # adaptador LoRA (pequeno)
tokenizer.save_pretrained('/content/drive/MyDrive/axon_ia_lora')
print('salvo em MyDrive/axon_ia_lora')

# (opcional) exportar em GGUF pra rodar offline no seu PC com Ollama/llama.cpp:
# model.save_pretrained_gguf('/content/drive/MyDrive/axon_ia_gguf', tokenizer, quantization_method='q4_k_m')

## Pronto! 🎉
Você tem sua IA em português. Para rodar **offline no seu PC**: exporte em GGUF (célula acima) e use com **Ollama** ou **llama.cpp**.

Para não alucinar em fatos, acople o seu **RAG do pyaxon** (router + `kb.json.gz`): recupere as passagens certas e passe como contexto no prompt antes de gerar.